# Infix-first penalty 48: NLLB source-embedding training

Train a **new 6,080 x 1,024 source embedding table** for the actual infix-first
penalty-48 tokenizer. Its first phase prioritizes recognized infixes;
the remaining vocabulary uses weighted morphology-aware BPE. There is no
additional root-first phase. The tokenizer is already trained and packaged.
The penalty changes its learned merge rules, not the translation loss.

All pretrained NLLB encoder layers, decoder, native Filipino target embeddings
and output projection stay frozen. Each candidate starts independently from
the same pinned NLLB base. Token IDs and embeddings from current32 or the other
candidate are incompatible and cannot be reused.

Matched recipe: seed42, native-subtoken warm initialization, learning rate
0.0003, batch1, accumulation8, maximum30 epochs, early stopping patience3 and
minimum improvement0.001. Source/target limits remain256/128. Selection uses
lowest validation loss, and saved history reports actual completed epochs.

Upload `infix_first_penalty48_training_inputs.zip` as a private Kaggle
dataset. It includes the exact matched CSV, tokenizer/runtime, native target
tokenizer, unchanged training helper and baseline provenance. It contains no
trained source weights. Enable GPU and Internet for dependencies and the
pinned base download. ZIP and automatically expanded inputs are supported.

Optional `current_weighted32_reference.zip` evaluates the currently installed
weighted32 model with the same rows and FP32 decoding. It is an inference
reference and is never used for initialization or resume. Run cells in order.
Both translation-evaluation switches default to False. The original notebook
and installed models are preserved. Local preparation did not run GPU training.


## 1. Install the recorded tokenizer/Transformers dependencies

Run this cell before importing Transformers. It retains the cloud runtime's GPU
PyTorch rather than installing a CPU wheel. If these packages were already imported
with different versions, restart the session after installation and continue.
As in the original notebook, remove optional vision/audio/UI packages before
installing the pinned stack. PyTorch itself is retained.
The baseline recorded Torch 2.11.0+cu128; a different GPU Torch version is recorded
as an environment difference rather than silently represented as identical.


In [ ]:
import sys, subprocess, os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
OPTIONAL_PACKAGES = ['diffusers', 'gradio', 'timm', 'torchvision', 'torchaudio']
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', *OPTIONAL_PACKAGES])
PINS = ['transformers==4.48.3', 'tokenizers==0.21.0',
        'huggingface-hub==0.28.1', 'sentencepiece==0.2.0',
        'safetensors==0.5.3', 'sacrebleu==2.5.1']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PINS])
print('Dependencies installed. Restart if a different version was already imported.')


## 2. Choose inputs and a new output run

Blank paths discover the matching Kaggle ZIP or expanded package. The notebook
distinguishes penalty16 and48 even when both are attached. Use `RESUME=False`
for a fresh run. Resume requires this exact candidate's full backup including
`last.pt`; current32, Plain BPE, the other penalty and inference-only ZIPs are
rejected. `RUN_NAME` must be a single folder name.

Kaggle working storage belongs to the session. Save/download outputs before
ending it. Checkpoints are saved at completed-epoch boundaries; an interrupted
epoch repeats when resumed. For the first run, keep all evaluation switches
False. Set `RUN_VALIDATION_TRANSLATIONS=True` to compare validation outputs.
Set `RUN_TEST=True` for final scoring after freezing the experiment settings.
Add the reference ZIP and enable `RUN_CURRENT32_REFERENCE` for a matched
current32 comparison. Refer to the included README for the full workflow.


In [ ]:
from pathlib import Path
import os, json, hashlib, shutil, zipfile, tempfile, importlib.util

INPUT_PATH = ''
DATA_CSV = ''
RUN_NAME = 'infix_first_penalty48_warm_start_seed42_v1'
RESUME = False
RESUME_BUNDLE = ''
RUN_VALIDATION_TRANSLATIONS = False
RUN_CURRENT32_REFERENCE = False
CURRENT32_REFERENCE_PATH = ''
RUN_TEST = False  # enable only after the experimental settings are frozen
USE_GOOGLE_DRIVE = True

EXPECTED_PACKAGE_SHA256 = 'd76effbf387e0be8d94a2f087f89b44f874bdd820974f261307fe7bf7acc27ca'
EXPECTED_PACKAGE_MANIFEST_SHA256 = 'a2d54df581751a5a1286e7ed543aacc7adbe31cd3799ef98119a594bf8e8bbbe'
EXPECTED_TOKENIZER_FINGERPRINT = 'fb606893cca542488118853aa58a5d97ebd7c02e51a1bdd9c54af09dc3625e86'
EXPECTED_DATA_SHA256 = '69a67a2773a5ecb49d16ef01c47057bd9b75916cf10f55d397df01d13c29c01b'
PENALTY = 48
VARIANT = 'infix_first_penalty48'
EXPECTED_REFERENCE_SHA256 = '558c2b95904eca7825c80afc7ca24c0958c623bb956fc1d5c77224a1baa6a520'
EXPECTED_REFERENCE_MANIFEST_SHA256 = 'd2e2887b7e31133901b2dea9889844c127b3be3933070a0b161baa029f724007'
PLATFORM = 'kaggle' if Path('/kaggle/working').exists() else 'colab'
if PLATFORM == 'kaggle':
    WORK = Path('/kaggle/working/infix_first_penalty48_training')
    SEARCH_ROOTS = [Path('/kaggle/input')]
    RUNS = WORK/'runs'
else:
    WORK = Path('/content/infix_first_penalty48_training')
    if USE_GOOGLE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        CLOUD = Path('/content/drive/MyDrive/kapampangan_nllb_infix_first_penalty48')
        SEARCH_ROOTS = [CLOUD, Path('/content')]
        RUNS = CLOUD/'runs'
    else:
        SEARCH_ROOTS = [Path('/content')]
        RUNS = WORK/'runs'
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in ('.', '..'):
    raise ValueError('RUN_NAME must be one folder name.')
WORK.mkdir(parents=True, exist_ok=True)
OUT = RUNS/'morph_bpe'/RUN_NAME
print('Platform:', PLATFORM, 'Output:', OUT)


## 3. Verify and unpack the exact inputs

Verification includes all package files, the infix-first penalty-48 fingerprint, tokenizer
inventory, baseline configuration and data identity. No existing trained source
weights or optimizer checkpoints are included in the training input package.


In [ ]:
def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def discover(pattern):
    result = set()
    for root in SEARCH_ROOTS:
        if root.exists():
            for p in root.rglob(pattern):
                if WORK not in p.parents and RUNS not in p.parents and p.is_file():
                    result.add(p.resolve())
    return sorted(result)

if not INPUT_PATH:
    candidates = discover('infix_first_penalty48_training_inputs.zip')
    if not candidates:
        candidates = [p.parent for p in discover('package_manifest.json')
                      if json.loads(p.read_text(encoding='utf-8')).get('package_kind') == 'infix_first_source_training'
                      and json.loads(p.read_text(encoding='utf-8')).get('tokenizer_variant') == VARIANT
                      and json.loads(p.read_text(encoding='utf-8')).get('crossing_penalty') == PENALTY]
    if len(candidates) != 1:
        raise ValueError('Set INPUT_PATH to the ZIP or extracted package folder. Found: '+str(candidates))
    INPUT_PATH = str(candidates[0])
package_path = Path(INPUT_PATH)
if package_path.is_file():
    if digest(package_path) != EXPECTED_PACKAGE_SHA256:
        raise ValueError('Training input ZIP hash mismatch. Use the supplied package.')
    INPUT_ROOT = WORK/'inputs'
    INPUT_ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(package_path) as archive:
        for entry in archive.infolist():
            target_path = (INPUT_ROOT/entry.filename).resolve()
            if not target_path.is_relative_to(INPUT_ROOT.resolve()):
                raise ValueError('Unsafe ZIP path')
            if (entry.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('ZIP links are not supported')
        archive.extractall(INPUT_ROOT)
else:
    INPUT_ROOT = package_path
if digest(INPUT_ROOT/'package_manifest.json') != EXPECTED_PACKAGE_MANIFEST_SHA256:
    raise ValueError('Package manifest hash mismatch. Use the supplied input files.')
package = json.loads((INPUT_ROOT/'package_manifest.json').read_text(encoding='utf-8'))
if (package['package_kind'] != 'infix_first_source_training' or
    package['tokenizer_variant'] != VARIANT or package['crossing_penalty'] != PENALTY):
    raise ValueError('Unexpected input package')
for name, checksum in package['files'].items():
    file = (INPUT_ROOT/name).resolve()
    if not file.is_relative_to(INPUT_ROOT.resolve()) or digest(file) != checksum:
        raise ValueError('Package checksum mismatch: '+name)
if (INPUT_ROOT/'last.pt').exists() or (INPUT_ROOT/'best_source.safetensors').exists():
    raise ValueError('Input package must not contain trained source weights.')
baseline = json.loads((INPUT_ROOT/'reference/plain_bpe/manifest.json').read_text(encoding='utf-8'))
baseline_audit = json.loads((INPUT_ROOT/'reference/plain_bpe/data_audit.json').read_text(encoding='utf-8'))
baseline_splits = json.loads((INPUT_ROOT/'reference/plain_bpe/split_manifest.json').read_text(encoding='utf-8'))
if baseline['identity']['data_sha256'] != EXPECTED_DATA_SHA256:
    raise ValueError('Baseline data identity mismatch')
ARTIFACT = INPUT_ROOT/'source_artifact'
artifact_manifest = json.loads((ARTIFACT/'tokenizer-manifest.json').read_text(encoding='utf-8'))
if artifact_manifest['artifact_fingerprint'] != EXPECTED_TOKENIZER_FINGERPRINT:
    raise ValueError('This is not the visualization tokenizer')
sys.path.insert(0, str(INPUT_ROOT))
from training_support import (variant_fields, validate_variant_artifact,
    validate_saved_manifest, validate_runtime_files, validate_checkpoint,
    validate_best_weights, identity_digest, reload_report_matches, extract_safe_zip)
VARIANT_FIELDS = validate_variant_artifact(artifact_manifest,
    json.loads((ARTIFACT/'tokenizer.json').read_text(encoding='utf-8')), PENALTY, EXPECTED_TOKENIZER_FINGERPRINT)
if package.get('trained_source_weights_included') is not False:
    raise ValueError('Training inputs must not contain trained source weights')
for line in (ARTIFACT/'checksums.sha256').read_text(encoding='utf-8').splitlines():
    if not line.strip(): continue
    checksum, name = line.split(maxsplit=1)
    file = (ARTIFACT/name.lstrip('*')).resolve()
    if not file.is_relative_to(ARTIFACT.resolve()) or digest(file) != checksum:
        raise ValueError('Tokenizer inventory mismatch: '+name)
if digest(INPUT_ROOT/'nllb_source_adapter.py') != baseline['identity']['helper_sha256']:
    raise ValueError('Training implementation differs from the recorded baseline')
print('Verified infix_first_penalty48 artifact, unchanged training helper and baseline provenance.')


## 4. Load the matched CSV without changing the comparison

The reader follows the original notebook's recovered CSV loader and requires
exact agreement with the installed baseline's CSV bytes, audit and ordered split
IDs. Source and target strings remain unchanged. It preserves accepted within-split
duplicates and evaluation references;
it does not silently deduplicate, resplit, or truncate the dataset.


In [ ]:
sys.path.insert(0, str(INPUT_ROOT))
sys.path.insert(0, str(INPUT_ROOT/'runtime'))
from training_support import read_matched_csv
if not DATA_CSV:
    packaged_csv = INPUT_ROOT/'data/kpm_tgl_cleaned.csv'
    candidates = [packaged_csv] if packaged_csv.is_file() else [p for p in discover('*.csv') if digest(p) == EXPECTED_DATA_SHA256]
    if len(candidates) != 1:
        raise ValueError('Set DATA_CSV to the original matching CSV. Hash-matched files: '+str(candidates))
    DATA_CSV = str(candidates[0])
splits, data_audit = read_matched_csv(DATA_CSV, EXPECTED_DATA_SHA256,
                                     baseline_audit, baseline_splits)
print('Matched rows:', {k:len(v) for k,v in splits.items()})
print('Preserved duplicates:', data_audit['within_split_duplicate_excess'])
print('Tokenizer-training-corpus overlap with translation test remains unverified.')


## 5. Load exact tokenization and reproduce baseline settings

Source content token pieces come from the exact visualization runtime/artifact.
The adapter adds BOS/EOS and swaps native IDs 0/1 at the NLLB boundary. These
control differences do not change the content segmentation. Tokenization limits
are checked before any training; overlength examples cause an error.


In [ ]:
import torch, transformers, random
import nllb_source_adapter as helper
from transformers import AutoTokenizer
if transformers.__version__ != '4.48.3':
    raise RuntimeError('Restart the session to use Transformers 4.48.3.')
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime. This notebook does not start CPU training.')
DEVICE = torch.device('cuda')
CFG = dict(baseline['cfg'])
expected_cfg = dict(seed=42, epochs=30, early_stopping_patience=3,
                    early_stopping_min_delta=0.001, lr=0.0003,
                    batch_size=1, accumulation=8, initialization='warm_start')
if CFG != expected_cfg:
    raise ValueError('Unexpected baseline training configuration')
MODEL_ID, REVISION = baseline['model_id'], baseline['revision']
SOURCE_LIMIT = baseline['identity']['source_limit']
TARGET_LIMIT = baseline['identity']['target_limit']
BEAMS = baseline['identity']['beams']
MAX_NEW_TOKENS = baseline['identity']['max_new_tokens']
random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])
torch.cuda.manual_seed_all(CFG['seed'])
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)
RUNTIME_CONTROLS = dict(gradient_checkpointing=True, checkpoint_use_reentrant=False,
    cudnn_benchmark=False, cuda_matmul_allow_tf32=False, cudnn_allow_tf32=False,
    deterministic_algorithms=True, deterministic_warn_only=True)
source = helper.SourceTokenizer(ARTIFACT, expected_size=6080)
target = AutoTokenizer.from_pretrained(INPUT_ROOT/'target_tokenizer',
                                     src_lang='tgl_Latn', tgt_lang='tgl_Latn')
encoded, tokenization_audit = helper.prepare(splits, source, target, SOURCE_LIMIT, TARGET_LIMIT)
for text in ['sinulat', 'kabukasan', 'magsalita', 'pipaglutuan', 'ing an ang', '']:
    if not text: continue
    native = source.backend.encode(text, add_special_tokens=False)
    if source.content_ids(text) != [source.to_model_id(i) for i in native.ids]:
        raise ValueError('Content token parity failure')
print('GPU:', torch.cuda.get_device_name(), 'Torch:', torch.__version__)
print('Baseline Torch: 2.11.0+cu128; record any environment difference.')
print(json.dumps(tokenization_audit, indent=2))


## 6. Prepare a separate run or verify an exact resume

Identity includes the actual infix-first algorithm, penalty, artifact, runtime,
helper, support modules, native target tokenizer, data, ordered splits and
configuration. A fresh run refuses existing weights. Application routing is
`morph_bpe`; research variant is recorded separately and cannot be confused
with current weighted32. No trained checkpoint supplies fresh initialization.


In [ ]:
identity = dict(baseline['identity'])
identity.update(**VARIANT_FIELDS)
identity.update(condition='morph_bpe', tokenizer_sha256=digest(ARTIFACT/'tokenizer.json'),
                artifact_manifest_sha256=digest(ARTIFACT/'tokenizer-manifest.json'),
                helper_sha256=digest(INPUT_ROOT/'nllb_source_adapter.py'),
                runtime_sha256=digest(INPUT_ROOT/'runtime/kapampangan_morphbpe_runtime/tokenizer.py'),
                tokenizer_variant='infix_first_penalty48', crossing_penalty=48,
                runtime_controls=RUNTIME_CONTROLS,
                tokenizer_fingerprint=EXPECTED_TOKENIZER_FINGERPRINT,
                input_package_manifest_sha256=EXPECTED_PACKAGE_MANIFEST_SHA256,
                target_tokenizer_sha256=digest(INPUT_ROOT/'target_tokenizer/tokenizer.json'),
                evaluation_support_sha256=digest(INPUT_ROOT/'evaluation_support.py'),
                training_support_sha256=digest(INPUT_ROOT/'training_support.py'),
                split_manifest_sha256=digest(INPUT_ROOT/'reference/plain_bpe/split_manifest.json'))
if RESUME_BUNDLE and not RESUME:
    raise ValueError('RESUME_BUNDLE requires RESUME=True')
if RESUME_BUNDLE:
    if OUT.exists():
        raise FileExistsError('Output exists; resume it directly or choose a new RUN_NAME.')
    resume_path = Path(RESUME_BUNDLE)
    if resume_path.is_file():
        RUNS.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=RUNS) as temporary:
            temp = Path(temporary)
            extract_safe_zip(resume_path, temp)
            info = json.loads((temp/'manifest.json').read_text(encoding='utf-8'))
            validate_saved_manifest(info, identity, CFG)
            validate_runtime_files(temp, identity)
            OUT.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(temp, OUT)
    else:
        info = json.loads((resume_path/'manifest.json').read_text(encoding='utf-8'))
        validate_saved_manifest(info, identity, CFG)
        validate_runtime_files(resume_path, identity)
        OUT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(resume_path, OUT, ignore=shutil.ignore_patterns('__pycache__'))
if RESUME:
    if not (OUT/'last.pt').is_file() or not (OUT/'best_source.safetensors').is_file():
        raise FileNotFoundError('Resume requires the full infix_first_penalty48 backup including last.pt and best weights.')
    saved = torch.load(OUT/'last.pt', map_location='cpu', weights_only=True)
    validate_saved_manifest(json.loads((OUT/'manifest.json').read_text(encoding='utf-8')), identity, CFG)
    validate_runtime_files(OUT, identity)
    validate_checkpoint(saved, identity, CFG, lambda tensor: torch.isfinite(tensor).all())
    from safetensors.torch import load_file
    saved_best = load_file(str(OUT/'best_source.safetensors'))['weight']
    validate_best_weights(saved_best, lambda tensor: torch.isfinite(tensor).all())
    del saved, saved_best
    print('Verified resume checkpoint.')
elif OUT.exists() and any(OUT.iterdir()):
    raise FileExistsError('Use a new RUN_NAME or explicitly resume this infix_first_penalty48 run.')
OUT.mkdir(parents=True, exist_ok=True)
manifest = dict(model_id=MODEL_ID, revision=REVISION, source_vocab_size=6080,
                source_format='kapampangan_project', source_id_mapping='swap_0_1',
                cfg=CFG, identity=identity, adaptation='source_embedding_only',
                target_language='tgl_Latn', source_marker='<s>',
                tokenizer_variant='infix_first_penalty48', weighted_penalty_used=True,
                crossing_penalty=48, training_status='prepared')
manifest.update(**VARIANT_FIELDS)
for name in ['source_artifact', 'runtime', 'target_tokenizer']:
    shutil.copytree(INPUT_ROOT/name, OUT/name, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('__pycache__'))
shutil.copy2(INPUT_ROOT/'nllb_source_adapter.py', OUT/'nllb_source_adapter.py')
shutil.copy2(INPUT_ROOT/'training_support.py', OUT/'training_support.py')
shutil.copy2(INPUT_ROOT/'evaluation_support.py', OUT/'evaluation_support.py')
helper.save_json(OUT/'manifest.json', manifest)
helper.save_json(OUT/'data_audit.json', data_audit)
shutil.copy2(INPUT_ROOT/'reference/plain_bpe/split_manifest.json', OUT/'split_manifest.json')
validate_runtime_files(OUT, identity)
helper.save_json(OUT/'tokenization_audit.json', tokenization_audit)
helper.save_json(OUT/'baseline_reference.json', baseline)
helper.save_json(OUT/'input_provenance.json', package)
environment = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(OUT/'environment.txt').write_text(environment, encoding='utf-8')
helper.save_json(OUT/'hardware.json', dict(torch=torch.__version__, gpu=torch.cuda.get_device_name(),
                 baseline_torch='2.11.0+cu128', python=sys.version,
                 runtime_controls=RUNTIME_CONTROLS))
print('Separate infix_first_penalty48 output prepared:', OUT)


## 7. Load the frozen base and initialize only the new source table

The base downloads from its pinned revision if it is not cached. No old source
embeddings are loaded for a fresh run. The preflight performs one temporary
backward/update check and restores the weights before training.
Gradient checkpointing uses the original non-reentrant setup to reduce GPU
activation memory. Training does not use the generation cache.


In [ ]:
model = helper.load_base(MODEL_ID, REVISION, CFG['seed'], 6080,
                         source=source, target=target,
                         initialization=CFG['initialization']).to(DEVICE)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
trainable = [(name, p.numel()) for name,p in model.named_parameters() if p.requires_grad]
if trainable != [('model.encoder.embed_tokens.weight', 6225920)]:
    raise ValueError('Only the source embedding table may train: '+str(trainable))
print('Trainable parameters:', trainable)
print('Preflight loss:', helper.preflight(model, encoded['train'][0], DEVICE))


## 8. Train the source embeddings and select by validation loss

This is the GPU training cell. It uses the unchanged helper from the recorded
30-epoch runs. Early stopping may stop before epoch 30; the lowest validation
loss selects the inference weights. Do not claim 30 completed epochs unless
the saved history actually contains them.


In [ ]:
import time
started = time.time()
history = helper.train(model, encoded, CFG, identity, OUT, DEVICE, resume=RESUME)
best = min(history, key=lambda row:row['validation_loss'])
from safetensors.torch import load_file
saved_weight = load_file(str(OUT/'best_source.safetensors'))['weight']
if tuple(saved_weight.shape) != (6080,1024) or not torch.isfinite(saved_weight).all():
    raise ValueError('Invalid exported source table')
if not torch.equal(saved_weight, model.get_encoder().embed_tokens.weight.detach().cpu()):
    raise ValueError('Model did not restore the selected best weights')
del saved_weight
manifest.update(training_status='completed', completed_epochs=history[-1]['epoch'],
                selected_epoch=best['epoch'], selected_validation_loss=best['validation_loss'])
helper.save_json(OUT/'manifest.json', manifest)
helper.save_json(OUT/'training_summary.json', dict(completed_epochs=history[-1]['epoch'],
                 selected_epoch=best['epoch'], best_validation_loss=best['validation_loss'],
                 configured_max_epochs=CFG['epochs'], this_session_seconds=time.time()-started,
                 tokenizer_variant='infix_first_penalty48', crossing_penalty=48))
print('Completed epochs:', history[-1]['epoch'], 'Selected epoch:', best['epoch'])
print('Best validation loss:', best['validation_loss'])
FULL_ZIP = Path(shutil.make_archive(str(OUT.parent/(RUN_NAME+'_full')), 'zip', root_dir=OUT))
print('Full epoch-boundary resume backup:', FULL_ZIP)


### 8.1 Reload the saved bundle and verify one validation translation

After completed training, reload the saved helper, source weights and tokenizers.
The first GPU model is freed before allocating the reloaded model. Check the
tokenizer IDs and selected weights, then generate one validation example in FP32
with four beams and 160 new tokens. This checks the saved bundle's loading path;
it is not a translation-quality measurement or a fresh-environment deployment test.

Interrupted training skips this check; section 10 can still export a full resume
backup. Completed inference export requires a passing check for the current weights.


In [ ]:
reload_info = json.loads((OUT/'manifest.json').read_text(encoding='utf-8'))
if reload_info.get('training_status') != 'completed':
    print('Training unfinished: bundle reload skipped. Section 10 can save the full resume backup.')
else:
    validate_saved_manifest(reload_info, identity, CFG)
    validate_runtime_files(OUT, identity)
    from safetensors.torch import load_file
    selected_weight = load_file(str(OUT/'best_source.safetensors'))['weight']
    if tuple(selected_weight.shape) != (6080,1024) or not torch.isfinite(selected_weight).all():
        raise ValueError('Invalid selected weights for bundle reload')
    example = splits['validation'][0]['raw_source']
    expected_ids = source.encode(example)
    if 'model' in globals():
        del model
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    for name in list(sys.modules):
        if name == 'kapampangan_morphbpe_runtime' or name.startswith('kapampangan_morphbpe_runtime.'):
            del sys.modules[name]
    sys.path.insert(0, str(OUT/'runtime'))
    saved_spec = importlib.util.spec_from_file_location('infix_first_penalty48_saved_adapter', OUT/'nllb_source_adapter.py')
    saved_helper = importlib.util.module_from_spec(saved_spec)
    saved_spec.loader.exec_module(saved_helper)
    model, source, target = saved_helper.load_bundle(OUT, DEVICE)
    if not torch.equal(selected_weight, model.get_encoder().embed_tokens.weight.detach().cpu()):
        raise ValueError('Reloaded source weights differ from the selected checkpoint')
    if source.size != 6080 or source.encode(example) != expected_ids:
        raise ValueError('Reloaded tokenizer IDs differ from the training tokenizer')
    if model.get_encoder().embed_tokens.weight.data_ptr() == model.get_decoder().embed_tokens.weight.data_ptr():
        raise ValueError('Reloaded source and native target embeddings are incorrectly tied')
    del selected_weight
    if len(expected_ids) > SOURCE_LIMIT:
        raise ValueError('Reload example exceeds the trained source limit')
    model.eval()
    x = torch.tensor([expected_ids], device=DEVICE)
    with torch.inference_mode():
        generated = model.generate(input_ids=x, attention_mask=x.ne(source.pad).long(),
            forced_bos_token_id=target.convert_tokens_to_ids('tgl_Latn'),
            max_new_tokens=MAX_NEW_TOKENS, num_beams=BEAMS, do_sample=False, use_cache=True)
    translation = target.decode(generated[0], skip_special_tokens=True)
    reload_report = dict(**VARIANT_FIELDS, identity_sha256=identity_digest(identity), status='passed', weights_sha256=digest(OUT/'best_source.safetensors'),
        tokenizer_sha256=identity['tokenizer_sha256'], selected_epoch=reload_info['selected_epoch'],
        base_revision=REVISION, beams=BEAMS, max_new_tokens=MAX_NEW_TOKENS, precision='fp32',
        validation_id=splits['validation'][0]['id'], source=example, prediction=translation)
    helper.save_json(OUT/'reload_check.json', reload_report)
    FULL_ZIP = Path(shutil.make_archive(str(OUT.parent/(RUN_NAME+'_full')), 'zip', root_dir=OUT))
    print('Saved bundle reload passed.')
    print('Kapampangan:', example)
    print('Filipino:', translation)


## 9. Optional candidate translation evaluation

Validation evaluation supports decisions. Keep final test scoring until both
candidate recipes are frozen. FP32 generation uses four beams and160 new
tokens, raw CSV references, BLEU13a and chrF++ with word order2. Progress
resumes only with the same weights, tokenizer, row order, settings and versions.
Outputs include predictions, scores, metric signatures and output-limit counts.
Loss is the training objective; BLEU/chrF++ are measured after training.


In [ ]:
from evaluation_support import evaluate_selected, verify_reference, compare_scores
if digest(INPUT_ROOT/'evaluation_support.py') != identity['evaluation_support_sha256']:
    raise ValueError('Evaluation implementation changed')
eval_splits = ([] if not RUN_VALIDATION_TRANSLATIONS else ['validation']) + ([] if not RUN_TEST else ['test'])
for split in eval_splits:
    evaluate_selected(model, source, target, splits[split], split, OUT/'evaluation'/split,
                      OUT, EXPECTED_DATA_SHA256, DEVICE, BEAMS, MAX_NEW_TOKENS, SOURCE_LIMIT)
if not eval_splits:
    print('Translation scoring skipped. Enable a switch and rerun after completed training.')


### 9.1 Optional matched comparison against current weighted32

Attach the reference ZIP and enable `RUN_CURRENT32_REFERENCE`. This scores
current32 on the same enabled partitions, using its own source embeddings
and tokenizer. The candidate GPU model is released before loading current32.
The CSV reports candidate-minus-current32 BLEU/chrF++; positive values mean
higher corpus scores. Differences alone do not establish significance.

To rerun candidate scoring afterward, rerun section8.1 to reload its bundle.


In [ ]:
if RUN_CURRENT32_REFERENCE:
    if not eval_splits:
        raise ValueError('Enable validation or final test scoring before current32 comparison')
    reference_bundle, reference_info = verify_reference(
        CURRENT32_REFERENCE_PATH, SEARCH_ROOTS, WORK,
        EXPECTED_REFERENCE_SHA256, EXPECTED_REFERENCE_MANIFEST_SHA256,
        EXPECTED_DATA_SHA256, REVISION, identity['helper_sha256'],
        identity['target_tokenizer_sha256'], CFG, baseline_splits)
    if 'model' in globals():
        del model
    import gc
    gc.collect(); torch.cuda.empty_cache()
    # Import the baseline's runtime from its verified files, even after candidate use.
    for name in list(sys.modules):
        if name == 'kapampangan_morphbpe_runtime' or name.startswith('kapampangan_morphbpe_runtime.'):
            del sys.modules[name]
    sys.path.insert(0, str(reference_bundle/'runtime'))
    ref_spec = importlib.util.spec_from_file_location('current32_verified_adapter', reference_bundle/'nllb_source_adapter.py')
    reference_helper = importlib.util.module_from_spec(ref_spec)
    ref_spec.loader.exec_module(reference_helper)
    reference_model, reference_source, reference_target = reference_helper.load_bundle(reference_bundle, DEVICE)
    reference_scores = WORK/'reference_evaluation'/'weighted_penalty32'
    for split in eval_splits:
        evaluate_selected(reference_model, reference_source, reference_target, splits[split], split,
            reference_scores/split, reference_bundle, EXPECTED_DATA_SHA256,
            DEVICE, BEAMS, MAX_NEW_TOKENS, SOURCE_LIMIT)
        compare_scores(OUT/'evaluation', reference_scores, split)
    del reference_model
    gc.collect(); torch.cuda.empty_cache()
else:
    print('Current32 comparison skipped. Its reference package is optional.')


## 10. Export inference and full resume backups

The inference ZIP has `manifest.json` at its root and excludes optimizer state.
It cannot resume training. Keep the full ZIP for an epoch-boundary resume. Both
contain the candidate's source tokenizer, native target tokenizer and helpers;
the frozen NLLB base is downloaded separately. Completed inference export
requires a passing saved-bundle reload check bound to these exact weights,
identity, selected epoch and decoding settings. An interrupted run can export
a full backup after at least one completed epoch.

Training creates separate files. Evaluate the candidates before deciding
whether to install one in the application. Installation is a separate step.


In [ ]:
info = json.loads((OUT/'manifest.json').read_text(encoding='utf-8'))
if not (OUT/'last.pt').is_file() or not (OUT/'best_source.safetensors').is_file():
    raise ValueError('At least one completed epoch is required for a full resume backup')
validate_saved_manifest(info, identity, CFG)
validate_runtime_files(OUT, identity)
reference_scores = WORK/'reference_evaluation'/'weighted_penalty32'
if reference_scores.exists():
    shutil.copytree(reference_scores, OUT/'evaluation'/'current_weighted32_reference', dirs_exist_ok=True)
FULL_ZIP = Path(shutil.make_archive(str(OUT.parent/(RUN_NAME+'_full')), 'zip', root_dir=OUT))
exports = [FULL_ZIP]
reload_path = OUT/'reload_check.json'
reload_ok = False
if reload_path.is_file():
    report = json.loads(reload_path.read_text(encoding='utf-8'))
    reload_ok = reload_report_matches(report, info, OUT)
if info.get('training_status') == 'completed' and reload_ok:
    INFERENCE_ZIP = OUT.parent/(RUN_NAME+'_inference.zip')
    with zipfile.ZipFile(INFERENCE_ZIP, 'w', zipfile.ZIP_DEFLATED) as archive:
        for file in sorted(OUT.rglob('*')):
            rel = file.relative_to(OUT)
            if not file.is_file() or '__pycache__' in rel.parts or file.suffix == '.pt': continue
            archive.write(file, str(rel).replace('\\','/'))
    exports.insert(0,INFERENCE_ZIP)
    print('Inference ZIP:', INFERENCE_ZIP)
else:
    print('Full resume backup saved; inference export needs completed training and a passing section 8.1 reload check.')
sums = OUT.parent/(RUN_NAME+'_SHA256SUMS.txt')
sums.write_text(''.join(digest(p)+'  '+p.name+'\n' for p in exports), encoding='utf-8')
print('Full resume ZIP:', FULL_ZIP)
print('Checksums:', sums)
from IPython.display import display, FileLink
for file in [*exports,sums]: display(FileLink(str(file)))
